Importing Libraries and datasets

In [ ]:
import pandas as pd

In [ ]:
sentiment = pd.read_csv("sentiment.csv")

Filtering so its only companies with at least 330 days of 5+ articles

In [ ]:
sentiment_data = (
    sentiment.assign(
        Date=pd.to_datetime(sentiment["date"]).dt.date,
        difference=sentiment["positive"] - sentiment["negative"]
    )

    .groupby(["company", "Date"])
    .filter(lambda x: len(x) >= 5)

    .groupby(["company", "Date"])[["positive", "negative", "difference"]]
    .mean()
    .reset_index()

    .groupby("company")
    .filter(lambda x: len(x) >= 330)

    .rename(columns={
        "positive": "Average Positive",
        "negative": "Average Negative",
        "difference": "Average Difference"
    })
)

Assigning each company a ticker to be used for later

In [ ]:
ticker_map = {
    "1 Automotive": "GPI",
    "3M": "MMM",
    "Airbnb": "ABNB",
    "American Airlines": "AAL",
    "American International": "AIG",
    "Andersons": "ANDE",
    "Apple": "AAPL",
    "Ball": "BALL",
    "Bank of America": "BAC",
    "Berkshire Hathaway": "BRK-B",
    "Best Buy": "BBY",
    "Caesars Entertainment": "CZR",
    "Campbell's": "CPB",
    "Capital One": "COF",
    "Carrier Global": "CARR",
    "Coca-Cola": "KO",
    "Crown": "CCK",
    "Cummins": "CMI",
    "Dana": "DAN",
    "Deere": "DE",
    "Dollar General": "DG",
    "Dover": "DOV",
    "Ebay": "EBAY",
    "Endeavor": "EDR",
    "FM": "FMCC",
    "Ferguson": "FERG",
    "Fox": "FOX",
    "Gap": "GAP",
    "General Electric": "GE",
    "Guardian Life": None,
    "HP": "HPQ",
    "Jacobs Solutions": "J",
    "Lockheed Martin": "LMT",
    "Lowe's": "LOW",
    "Marriott International": "MAR",
    "McDonald's": "MCD",
    "Mosaic": "MOS",
    "Murphy USA": "MUSA",
    "Nike": "NKE",
    "Nordstrom": "JWN",
    "Northrop Grumman": "NOC",
    "Nvidia": "NVDA",
    "Oscar Health": "OSCR",
    "PayPal": "PYPL",
    "Post": "POST",
    "Progressive": "PGR",
    "RTX": "RTX",
    "Reliance": "RS",
    "Sirius XM": "SIRI",
    "Southern": "SO",
    "Southwest Airlines": "LUV",
    "State Farm": None,
    "Tesla": "TSLA",
    "Travelers": "TRV",
    "Uber": "UBER",
    "United Airlines": "UAL",
    "Walmart": "WMT",
    "Walt Disney": "DIS",
    "Williams": "WMB"
}

sentiment_data["ticker"] = sentiment_data["company"].map(ticker_map)

sentiment_data = sentiment_data.dropna(subset=["ticker"])

Creating a dataframe of ticker and closing price

In [ ]:
import yfinance as yf

sentiment_data["Date"] = pd.to_datetime(sentiment_data["Date"])

tickers = sentiment_data["ticker"].dropna().unique().tolist()

bad_tickers = ["JWN", "EDR"]
tickers = [t for t in tickers if t not in bad_tickers]

price_data = yf.download(
    tickers=tickers,
    start=sentiment_data["Date"].min(),
    end=sentiment_data["Date"].max(),
    group_by="ticker",
    auto_adjust=False,
    progress=True
)

price_list = []

for ticker in tickers:
    try:
        df = price_data[ticker][["Close"]].reset_index()
        df["ticker"] = ticker
        df = df.rename(columns={"Close": "Close Price"})
        price_list.append(df)
    except:
        print(f"Skipping {ticker}")

price_df = pd.concat(price_list, ignore_index=True)

price_df["Date"] = pd.to_datetime(price_df["Date"])

final_df = sentiment_data.merge(
    price_df,
    on=["ticker", "Date"],
    how="left"
)

final_df = final_df.sort_values(["ticker", "Date"])
final_df["Close Price"] = final_df.groupby("ticker")["Close Price"].ffill()

/tmp/ipykernel_10270/3600712065.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentiment_data["Date"] = pd.to_datetime(sentiment_data["Date"])
[*********************100%***********************]  55 of 55 completed


1 Day lag Regression

In [ ]:
final_df = final_df.sort_values(["ticker", "Date"])

final_df["next_close"] = final_df.groupby("ticker")["Close Price"].shift(-1)

final_df["next_return"] = (
    (final_df["next_close"] - final_df["Close Price"])
    / final_df["Close Price"]
)

from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/sentiment_data.csv'

final_df.to_csv(file_path, index=False)

In [ ]:
final_df = pd.read_csv("sentiment_data.csv")

In [ ]:
import statsmodels.api as sm

def regression_model(df, lag):
  final_df = df.sort_values(["ticker", "Date"])

  final_df["next_close"] = final_df.groupby("ticker")["Close Price"].shift(-lag)

  final_df["next_return"] = (
      (final_df["next_close"] - final_df["Close Price"])
      / final_df["Close Price"]
  )
  results = []

  for ticker in final_df["ticker"].unique():
      df = final_df[final_df["ticker"] == ticker].dropna(
          subset=["Average Difference", "next_return"]
      )

      if len(df) < 10:
          continue

      X = df["Average Difference"]
      y = df["next_return"]

      X = sm.add_constant(X)

      model = sm.OLS(y, X).fit()

      results.append({
          "ticker": ticker,
          "coef": model.params["Average Difference"],
          "p_value": model.pvalues["Average Difference"],
          "r_squared": model.rsquared,
          "n_obs": len(df)
      })

  regression_results = pd.DataFrame(results)
  return regression_results

regression_results = regression_model(final_df,1)

In [ ]:
regression_results.sort_values(by = "r_squared")

,ticker,coef,p_value,r_squared,n_obs
24,GAP,0.000097,0.991721,3.267822e-07,332
40,NVDA,0.000662,0.922184,2.922434e-05,329
3,AIG,-0.000524,0.895082,5.326428e-05,329
34,MCD,-0.000409,0.878739,6.815915e-05,344
16,DE,-0.000657,0.864801,8.851658e-05,330
27,HPQ,-0.001006,0.856849,9.527945e-05,344
2,ABNB,-0.001154,0.842757,1.194163e-04,332
26,GPI,-0.000892,0.837029,1.272419e-04,335
46,RTX,0.000788,0.811919,1.728728e-04,330
0,AAL,0.002125,0.790526,2.134660e-04,333


2 day lag regression

In [ ]:
regression_results_2 = regression_model(final_df,2)
regression_results_2.sort_values(by="p_value")

,ticker,coef,p_value,r_squared,n_obs
20,EBAY,0.020647,0.008261,0.020824,334
35,MMM,0.009412,0.054610,0.010950,338
24,GAP,-0.022597,0.068729,0.010035,331
51,UAL,0.016797,0.073212,0.009549,337
4,ANDE,0.019154,0.100179,0.008219,330
26,GPI,-0.009629,0.103551,0.007962,334
53,WMB,-0.013269,0.119282,0.007294,334
30,LMT,-0.006181,0.127293,0.006969,335
18,DIS,0.012294,0.130235,0.006925,332
45,RS,-0.006641,0.137378,0.006676,332


3 day lag regression

In [ ]:
regression_results_3 = regression_model(final_df,3)
regression_results_3.sort_values(by="p_value")

,ticker,coef,p_value,r_squared,n_obs
20,EBAY,0.023991,0.010581,1.957708e-02,333
16,DE,-0.015754,0.010729,1.980026e-02,328
24,GAP,-0.033930,0.020726,1.620214e-02,330
29,KO,-0.007903,0.046188,1.202668e-02,331
35,MMM,0.011075,0.069769,9.782712e-03,337
3,AIG,0.011200,0.101005,8.254176e-03,327
47,SIRI,-0.019148,0.107794,7.865902e-03,330
4,ANDE,0.021452,0.130075,6.994122e-03,329
53,WMB,-0.015042,0.142880,6.474128e-03,333
34,MCD,0.005840,0.171643,5.488646e-03,342


1 week lag regression

In [ ]:
regression_results_7 = regression_model(final_df,7)
regression_results_7.sort_values(by="p_value")

,ticker,coef,p_value,r_squared,n_obs
20,EBAY,0.035008,0.008734,2.084060e-02,329
16,DE,-0.021840,0.011040,1.988917e-02,324
41,OSCR,0.077776,0.015663,1.783680e-02,327
46,RTX,-0.016219,0.037654,1.334891e-02,324
8,BRK-B,-0.014683,0.053243,1.144787e-02,327
34,MCD,0.011869,0.053770,1.102615e-02,338
39,NOC,-0.015013,0.053884,1.145719e-02,325
45,RS,-0.013844,0.068997,1.013846e-02,327
9,CARR,-0.018396,0.080473,9.458294e-03,324
24,GAP,-0.035748,0.106844,8.005768e-03,326


Comparing to Baseline LSTM Models

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def compare_lstm_models(df, sequence_length=10, lag=1):

    results = []

    df = df.copy()
    df = df.sort_values(["ticker", "Date"])

    df["return"] = df.groupby("ticker")["Close Price"].pct_change()
    df["target"] = df.groupby("ticker")["return"].shift(-lag)

    df["sentiment"] = df.groupby("ticker")["Average Difference"].shift(1)

    model_configs = {
        "price_only": ["return"],
        "sentiment_only": ["sentiment"],
        "combined": ["return", "sentiment"]
    }

    for ticker in df["ticker"].unique():
        sub = df[df["ticker"] == ticker].copy()

        for model_name, features in model_configs.items():

            sub_clean = sub.dropna(subset=features + ["target"])

            if len(sub_clean) < sequence_length + 20:
                continue

            X, y = [], []

            values = sub_clean[features].values
            targets = sub_clean["target"].values

            for i in range(len(sub_clean) - sequence_length):
                X.append(values[i:i+sequence_length])
                y.append(targets[i+sequence_length])

            X = np.array(X)
            y = np.array(y)

            scaler = StandardScaler()
            X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)

            split = int(0.8 * len(X))
            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            if len(X_test) == 0:
                continue

            model = Sequential([
                LSTM(32, input_shape=(sequence_length, X.shape[2])),
                Dropout(0.2),
                Dense(1)
            ])

            model.compile(optimizer="adam", loss="mse")

            model.fit(
                X_train, y_train,
                epochs=5,
                batch_size=16,
                verbose=0
            )

            preds = model.predict(X_test, verbose=0).flatten()

            mse = mean_squared_error(y_test, preds)
            rmse = np.sqrt(mse)

            results.append({
                "ticker": ticker,
                "model": model_name,
                "mse": mse,
                "rmse": rmse,
                "n_obs": len(sub_clean)
            })

    results_df = pd.DataFrame(results)

    comparison = results_df.pivot(index="ticker", columns="model", values="rmse")

    comparison["sentiment_vs_price"] = (
        comparison["price_only"] - comparison["sentiment_only"]
    )

    comparison["combined_vs_price"] = (
        comparison["price_only"] - comparison["combined"]
    )

    return comparison.reset_index()

1 day lag

In [ ]:
comparison_df = compare_lstm_models(
    final_df,
    sequence_length=7,
    lag=1
)

/tmp/ipykernel_549/906203793.py:14: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df.groupby("ticker")["Close Price"].pct_change()
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: U

In [ ]:
comparison_df.sort_values(by="sentiment_vs_price")

from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/1_day_lag.csv'

comparison_df.to_csv(file_path, index=False)

Mounted at /content/drive


2 day lag

In [ ]:
comparison_df2 = compare_lstm_models(
    final_df,
    sequence_length=7,
    lag=2
)

/tmp/ipykernel_549/906203793.py:14: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df.groupby("ticker")["Close Price"].pct_change()
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: U

In [ ]:
#comparison_df2.sort_values(by="sentiment_vs_price")

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/2_day_lag.csv'

comparison_df2.to_csv(file_path, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


3 day lag

In [ ]:
comparison_df3 = compare_lstm_models(
    final_df,
    sequence_length=7,
    lag=3
)

/tmp/ipykernel_549/906203793.py:14: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df.groupby("ticker")["Close Price"].pct_change()
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: U

In [ ]:
# comparison_df3.sort_values(by="sentiment_vs_price")

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/3_day_lag.csv'

comparison_df3.to_csv(file_path, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1 week lag

In [ ]:
comparison_df7 = compare_lstm_models(
    final_df,
    sequence_length=7,
    lag=7
)

/tmp/ipykernel_549/906203793.py:14: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df.groupby("ticker")["Close Price"].pct_change()
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: U

In [ ]:
#comparison_df7.sort_values(by="sentiment_vs_price")

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/7_day_lag.csv'

comparison_df7.to_csv(file_path, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
